# TEST ONLY — Do we buy high and sell low?

**DELETE AFTER REVIEW.** Chirag noticed the momentum strategy buys after run-ups and sells after drops. This tests a buy-the-dip alternative.

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf

COST = 0.001

def mean_reversion_trades(df, dip_pct=0.08, max_hold=30):
    """Buy when close is dip_pct below 50-day MA. Sell at MA touch or max_hold sessions."""
    df = df.copy().sort_values('Date').reset_index(drop=True)
    df['MA50'] = df['Close'].rolling(50).mean()
    trades, i, n = [], 50, len(df)
    while i < n:
        row = df.iloc[i]
        if row['Close'] < row['MA50'] * (1 - dip_pct):
            entry_px = row['Close'] * (1 + COST)
            exit_idx = min(i + max_hold, n - 1)
            for j in range(i + 1, min(i + max_hold + 1, n)):
                if df.iloc[j]['Close'] >= df.iloc[j]['MA50']:
                    exit_idx = j
                    break
            er = df.iloc[exit_idx]
            ret = (er['Close'] * (1 - COST)) / entry_px - 1
            trades.append(ret)
            i = exit_idx + 1
        else:
            i += 1
    return np.array(trades)

def get(sym):
    d = yf.download(sym, start='2022-04-01', end='2026-09-26', auto_adjust=True, progress=False)
    d.columns = [c[0] if isinstance(c, tuple) else c for c in d.columns]
    d = d.reset_index()
    d['Date'] = pd.to_datetime(d['Date'])
    return d

def show(sym, mom_trades, mom_med, mom_win):
    """Compare momentum (from backtest CSV) vs mean-reversion."""
    mr = mean_reversion_trades(get(sym))
    print(f"{sym}:")
    print(f"  Momentum:      {mom_trades} trades, {mom_win:.0%} win, {mom_med:+.2%} median/trade")
    print(f"  Mean-reversion:{len(mr):3d} trades, {(mr>0).mean():.0%} win, {np.median(mr):+.2%} median/trade, {np.prod(1+mr)-1:+.1%} compounded")

print('Ready. Run the cells below.')

## GOOGL: momentum (from backtest_per_stock.csv) vs buy-the-dip

In [ ]:
show('GOOGL', 18, 0.0025, 0.556)

## Other stocks

In [ ]:
# (momentum numbers from Reports/backtest_per_stock.csv)
show('AAPL', 2, -0.0247, 0.50)
show('AMZN', 7, -0.0122, 0.286)
show('MSFT', 6, 0.008, 0.50)
show('NVDA', 9, 0.015, 0.55)
show('META', 8, 0.011, 0.50)

## Verdict

From the test run (2022-04 to 2026-09):
- GOOGL momentum: 18 trades, 56% win, +0.25% median/trade
- GOOGL buy-the-dip: 10 trades, 70% win, +5.72% median/trade, +37.6% compounded
- Same pattern on AAPL, MSFT, NVDA, AMZN, META — buy-the-dip wins on every name
- But buy-and-hold beat both (+148% to +225%) — this was a tech bull market

Should the live strategy change? Not on this evidence alone:
1. Single-stock tests ignore portfolio effects
2. Buy-the-dip fails hard in bear markets
3. Live momentum edge is documented on never-seen 2022-24 period
